In [ ]:

# ============================================================
# TP2 - PARTE I: REGRESIÓN LOGÍSTICA CON REGULARIZACIÓN
# Inciso 1 - Preparación de datos
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Definimos las rutas del proyecto.
raiz = Path.cwd()

if raiz.name == "Programs":
    raiz = raiz.parent

carpeta_input = raiz / "Input"
carpeta_output = raiz / "Output"
carpeta_output.mkdir(exist_ok=True)

# Importamos la base limpia del TP1.
base = pd.read_csv(carpeta_input / "EPH_T126_limpia.csv")

# Recuperamos la partición original.
train = base.loc[base["conjunto"] == "train"].copy()
test = base.loc[base["conjunto"] == "test"].copy()

# Excluimos la dependiente, el indicador de partición
# y la columna de unos incorporada en el TP1.
columnas_excluir = ["informal", "conjunto", "const"]

X_train = train.drop(columns=columnas_excluir)
X_test = test.drop(columns=columnas_excluir)

y_train = train["informal"].astype(int)
y_test = test["informal"].astype(int)

# Estandarizamos utilizando exclusivamente entrenamiento.
scaler = StandardScaler()

X_train_sc = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_sc = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Verificaciones.
assert X_train.columns.equals(X_test.columns)
assert not X_train_sc.isna().any().any()
assert X_train_sc.shape == (13692, 68)

print("X_train:", X_train_sc.shape)
print("X_test:", X_test_sc.shape)
print("Cantidad de predictores:", X_train_sc.shape[1])


In [ ]:

# ============================================================
# Inciso 1 - Trayectorias de coeficientes LASSO y Ridge
# ============================================================

# Grilla de penalidades: lambda = 10^n, n = -5,...,5.
lambdas = 10.0 ** np.arange(-5, 6)

# Listas para almacenar los coeficientes.
coefs_lasso = []
coefs_ridge = []

# LASSO: penalización L1.
for lam in lambdas:

    lasso = LogisticRegression(
        penalty="l1",
        C=1/lam,
        solver="liblinear",
        max_iter=5000,
        random_state=444
    )

    lasso.fit(X_train_sc, y_train)
    coefs_lasso.append(lasso.coef_[0])

# Ridge: penalización L2.
for lam in lambdas:

    ridge = LogisticRegression(
        penalty="l2",
        C=1/lam,
        solver="liblinear",
        max_iter=5000,
        random_state=444
    )

    ridge.fit(X_train_sc, y_train)
    coefs_ridge.append(ridge.coef_[0])

# Convertimos los coeficientes en matrices.
coefs_lasso = np.array(coefs_lasso)
coefs_ridge = np.array(coefs_ridge)

# Guardamos los resultados para su posterior revisión.
tabla_lasso = pd.DataFrame(
    coefs_lasso, index=lambdas, columns=X_train.columns
)
tabla_ridge = pd.DataFrame(
    coefs_ridge, index=lambdas, columns=X_train.columns
)

tabla_lasso.index.name = "lambda"
tabla_ridge.index.name = "lambda"

tabla_lasso.to_csv(carpeta_output / "coeficientes_lasso.csv")
tabla_ridge.to_csv(carpeta_output / "coeficientes_ridge.csv")

print("Grilla de penalidades:", lambdas)
print("Dimensiones de cada matriz:", coefs_lasso.shape)


In [ ]:

# ============================================================
# Inciso 1 - Gráfico de coeficientes
# ============================================================

fig, axes = plt.subplots(
    1, 2, figsize=(13, 4.8), sharey=True
)

# Panel A: LASSO.
axes[0].plot(
    lambdas, coefs_lasso,
    linewidth=0.9, alpha=0.6
)
axes[0].set_xscale("log")
axes[0].set_title("Panel A. LASSO (L1)")
axes[0].set_xlabel(r"Penalidad $\lambda$")
axes[0].set_ylabel("Coeficientes estimados")
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].grid(axis="x", alpha=0.25)

# Panel B: Ridge.
axes[1].plot(
    lambdas, coefs_ridge,
    linewidth=0.9, alpha=0.6
)
axes[1].set_xscale("log")
axes[1].set_title("Panel B. Ridge (L2)")
axes[1].set_xlabel(r"Penalidad $\lambda$")
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()

# Exportamos automáticamente la figura.
fig.savefig(
    carpeta_output / "regularizacion_coeficientes.pdf",
    bbox_inches="tight"
)

fig.savefig(
    carpeta_output / "regularizacion_coeficientes.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()
